## Analysis Notebook 
This is where all the analysis takes place, from the zonal statisitics, pixel overlapping, to stream filtering. 

### Pixel agreement / disagreement 

This part is to analyze pixel by pixel the 3 scenarios we are interested in -- landsat only (sentinel dry, landsat inundated), sentinel only (landsat dry, sentinel inundated), and both inundated. 

Below is the function created to do each scenario and the code to run to get the 3 raster files. 

In [1]:
import rasterio
import numpy as np

def agree_disagree_mask(sentinel_path, landsat_path, output_path, chunk_size = 5000, mask_type = "both inundated"):
        """
        Creates an agreement or disagreement mask from Sentinel and Landsat rasters and saves as geoTiff

        Parameters: 
        - sentinel_path: str, path to Sentinel Raster 
        - landsat_path: str, path to Landsat Raster 
        - output_path: str, path to save the agreement mask raster 
        - chunk_size: int, number of rows to process at a time (default 5000)
        - mask_type: str, type of agreement or disagreement wanted (3 choices, both inundated, landsat only, sentinel only)
        """

        with rasterio.open(sentinel_path) as sentinel_raster, rasterio.open(landsat_path) as landsat_raster: 
            # Ensure rasters match in dimensions and transform
            assert sentinel_raster.shape == landsat_raster.shape, "Raster dimensions must match!"

            #copy metadata for output file
            meta = sentinel_raster.meta.copy()
            meta.update(dtype = rasterio.uint8, compress ='lzw')

            #Open output file to write in chunks 
            with rasterio.open(output_path, 'w', **meta) as dst:
                height, width = sentinel_raster.shape #get dimensions 

                for row_start in range(0, height, chunk_size):
                    row_end = min(row_start + chunk_size, height) #define chunk bounds 

                    #read the chunk for both rasters 
                    sentinel_chunk = sentinel_raster.read(1, window=((row_start, row_end), (0, width)))
                    landsat_chunk = landsat_raster.read(1, window=((row_start, row_end), (0, width)))

                    if mask_type == 'both inundated':
                        #both sentinel and landsat are inundated 
                        sentinel_mask = (sentinel_chunk == 1) 
                        landsat_mask = (landsat_chunk == 1) 
                    elif mask_type == 'sentinel only': 
                        #landsat is dry but sentinel is inudated 
                        sentinel_mask = (sentinel_chunk == 1) 
                        landsat_mask = (landsat_chunk == 0) 
                    elif mask_type == 'landsat only':
                        #sentinel is dry but landsat is inundated 
                        sentinel_mask = (sentinel_chunk == 0) 
                        landsat_mask = (landsat_chunk == 1) 
                    else:
                        raise ValueError(f"Invalid mask type: {mask_type}. Choose from 'both inundated', 'landsat only', 'sentinel only'.")
                    
                    #compute where there is a cross over in the masks 
                    mask = sentinel_mask & landsat_mask
                    #coverts to uint8 for storage (1 = intersection, 0 = no intersection)
                    mask = mask.astype(np.uint8)
                    # Write chunk to output file
                    dst.write(mask, 1, window=((row_start, row_end), (0, width)))

                    print(f"Processed rows {row_start} to {row_end}")

            print(f"Agreement mask saved at: {output_path}")

In [41]:
agree_disagree_mask(sentinel_path='../data/zenodo-data/sentinel_extent_inundation.tif',
                    landsat_path= '../data/zenodo-data/landsat_mode_inundation.tif',
                    output_path='../data/zenodo-data/both_inundated.tif',
                    chunk_size=5000, mask_type='both inundated')

KeyboardInterrupt: 

In [ ]:
agree_disagree_mask(sentinel_path='../data/zenodo-data/sentinel_extent_inundation.tif',
                    landsat_path= '../data/zenodo-data/landsat_mode_inundation.tif',
                    output_path='../data/zenodo-data/sentinel_only.tif',
                    chunk_size=5000, mask_type='sentinel only')

Processed rows 0 to 5000
Processed rows 5000 to 10000
Processed rows 10000 to 15000
Processed rows 15000 to 20000
Processed rows 20000 to 25000
Processed rows 25000 to 30000
Processed rows 30000 to 35000
Processed rows 35000 to 40000
Processed rows 40000 to 45000
Processed rows 45000 to 50000
Processed rows 50000 to 55000
Processed rows 55000 to 60000
Processed rows 60000 to 65000
Processed rows 65000 to 70000
Processed rows 70000 to 75000
Processed rows 75000 to 80000
Processed rows 80000 to 85000
Processed rows 85000 to 90000
Processed rows 90000 to 95000
Processed rows 95000 to 100000
Processed rows 100000 to 105000
Processed rows 105000 to 110000
Processed rows 110000 to 115000
Processed rows 115000 to 120000
Processed rows 120000 to 125000
Processed rows 125000 to 130000
Processed rows 130000 to 135000
Processed rows 135000 to 140000
Processed rows 140000 to 145000
Processed rows 145000 to 150000
Processed rows 150000 to 155000
Processed rows 155000 to 160000
Processed rows 160000

In [ ]:
agree_disagree_mask(sentinel_path='../data/zenodo-data/sentinel_extent_inundation.tif',
                    landsat_path= '../data/zenodo-data/landsat_mode_inundation.tif',
                    output_path='../data/zenodo-data/landsat_only.tif',
                    chunk_size=5000, mask_type='landsat only')

Processed rows 0 to 5000
Processed rows 5000 to 10000
Processed rows 10000 to 15000
Processed rows 15000 to 20000
Processed rows 20000 to 25000
Processed rows 25000 to 30000
Processed rows 30000 to 35000
Processed rows 35000 to 40000
Processed rows 40000 to 45000
Processed rows 45000 to 50000
Processed rows 50000 to 55000
Processed rows 55000 to 60000
Processed rows 60000 to 65000
Processed rows 65000 to 70000
Processed rows 70000 to 75000
Processed rows 75000 to 80000
Processed rows 80000 to 85000
Processed rows 85000 to 90000
Processed rows 90000 to 95000
Processed rows 95000 to 100000
Processed rows 100000 to 105000
Processed rows 105000 to 110000
Processed rows 110000 to 115000
Processed rows 115000 to 120000
Processed rows 120000 to 125000
Processed rows 125000 to 130000
Processed rows 130000 to 135000
Processed rows 135000 to 140000
Processed rows 140000 to 145000
Processed rows 145000 to 150000
Processed rows 150000 to 155000
Processed rows 155000 to 160000
Processed rows 160000

### stream segment filtering 

In [ ]:
#this cell is necessary to not get any errors when loading shapefiles 
import os
os.environ["PROJ_DATA"] = "/Users/ppuente/github/detection_comparison-CRB/comparison-env/lib/python3.11/site-packages/fiona/proj_data"
# replace the /Users/ppuente/github/detection_comparison-CRB/comparison-env to your own path and what you named the your environment (ie. comparison-env)

In [51]:
from pathlib import Path 
import re
import geopandas as gpd
from rasterstats import zonal_stats
import pandas as pd

folder_path = Path('../data/zenodo-data/crb_huc4_stream_flowlines')

shp_files = [str(shp_file) for shp_file in folder_path.glob('*flowlines.shp')] #list of all shapefile flowlines 

sentinel_raster = '../data/zenodo-data/sentinel_extent_inundation.tif'
landsat_raster = '../data/zenodo-data/landsat_mode_inundation.tif'

summary_list = []

In [52]:
for file in shp_files:
    hucid = re.findall(r"\d+", file)[2]
    
    huc_file =  gpd.read_file(file)
    huc_file = huc_file.drop(columns=['permanent_', 'nhdplusid', 'reachcode', 'fcode', 'gnis_name', 'wbarea_per', 'visibili_1', 'mainpath_d','mainpath'], axis=1)

    ## Single buffer size of 100 meters for all stream orders 
    huc_file['geometry'] = huc_file.geometry.buffer(100)

    print(f'initiating stats for {hucid}')

    stats = zonal_stats(huc_file, landsat_raster, ###!! CHANGE sentinel_raster to landsat_raster to run it for Landsat or vice versa
                        stats=['count'],
                        categorical=True, nodata=0)

    print(f'stats for {hucid} ran successfully')

    # add water pixel counts
    huc_file['inun_pixel'] = [s.get(1, 0) for s in stats]

    # -------------------------------------------------------------------------
    # (1) total counts (before filtering)
    total_grouped = huc_file.groupby("streamorde").agg(
        stream_segment_total=("streamorde", "size")
    ).reset_index()

    # (2) filtered counts (after filtering)
    streams_with_inundation = huc_file[huc_file['inun_pixel'] > 0]
    filtered_grouped = streams_with_inundation.groupby("streamorde").agg(
        stream_segment_count=("streamorde", "size"),
        total_water_pixel_count=("inun_pixel", "sum")
    ).reset_index()

    # (3) merge them into a single summary
    combined = pd.merge(
        total_grouped,
        filtered_grouped,
        on="streamorde",
        how="left"
    ).fillna(0)  # fill NaN with 0 where no filtered streams exist

    combined["huc_id"] = hucid

    # Append combined results
    summary_list.append(combined)

    #streams need to be saved 
    #!! COMMENT this when running for Landsat
    #streams_with_inundation.to_file(f"../data/zenodo-data/shapefiles/sentinel-filtered_stream_segments/sentinel_huc4_{hucid}_filtered.shp", driver= "ESRI Shapefile")

    #!! UNCOMMENT this for Landsat
    streams_with_inundation.to_file(f"../data/zenodo-data/shapefiles/landsat-filtered_stream_segments/landsat_huc4_{hucid}_filtered.shp", driver= "ESRI Shapefile")

    print(f'huc{hucid} filtered shapefile saved')

initiating stats for 1403
stats for 1403 ran successfully
huc1403 filtered shapefile saved


/Users/ppuente/github/detection_comparison-CRB/comparison-env/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field fdate create as date field, though DateTime requested.
  ogr_write(


initiating stats for 1407
stats for 1407 ran successfully
huc1407 filtered shapefile saved


/Users/ppuente/github/detection_comparison-CRB/comparison-env/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field fdate create as date field, though DateTime requested.
  ogr_write(


initiating stats for 1404
stats for 1404 ran successfully


...

/Users/ppuente/github/detection_comparison-CRB/comparison-env/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field fdate create as date field, though DateTime requested.
  ogr_write(


huc1401 filtered shapefile saved
initiating stats for 1406
stats for 1406 ran successfully


/Users/ppuente/github/detection_comparison-CRB/comparison-env/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field fdate create as date field, though DateTime requested.
  ogr_write(


huc1406 filtered shapefile saved
initiating stats for 1402
stats for 1402 ran successfully
huc1402 filtered shapefile saved


/Users/ppuente/github/detection_comparison-CRB/comparison-env/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field fdate create as date field, though DateTime requested.
  ogr_write(


In [53]:
summary_df = pd.concat(summary_list, ignore_index=True)

In [54]:
summary_df

,streamorde,stream_segment_total,stream_segment_count,total_water_pixel_count,huc_id
0,1,29490,1323,167951,1403
1,2,13307,710,115491,1403
2,3,6524,368,67354,1403
3,4,3373,190,51972,1403
4,5,1627,77,30260,1403
...,...,...,...,...,...
126,4,4505,260,67808,1402
127,5,2166,176,81295,1402
128,6,1253,187,83378,1402
129,7,851,291,39955,1402


In [55]:
summary_df['detection_ratio'] = summary_df['stream_segment_count'] / summary_df['stream_segment_total']
summary_df['detection_perc'] = round((summary_df['stream_segment_count']/summary_df['stream_segment_total'])*100, 2)

In [56]:
summary_df[summary_df['streamorde']==2]

,streamorde,stream_segment_total,stream_segment_count,total_water_pixel_count,huc_id,detection_ratio,detection_perc
1,2,13307,710,115491,1403,0.053355,5.34
10,2,11007,559,287816,1407,0.050786,5.08
19,2,27057,3230,663396,1404,0.119378,11.94
27,2,30576,824,84019,1504,0.026949,2.69
35,2,28938,1269,401701,1503,0.043852,4.39
45,2,25833,340,57129,1507,0.013161,1.32
54,2,24201,1417,272596,1506,0.058551,5.86
62,2,29354,1875,205193,1502,0.063875,6.39
70,2,41781,2433,302926,1408,0.058232,5.82
79,2,33269,632,54832,1505,0.018997,1.90


In [57]:
#save the data as csv
summary_df.to_csv('../data/landsat_stream_count_100m.csv', index=False)  

## Zonal Statistics

In [58]:
#this cell is necessary to not get any errors when loading shapefiles 
import os
os.environ["PROJ_DATA"] = "/Users/ppuente/github/detection_comparison-CRB/comparison-env/lib/python3.11/site-packages/fiona/proj_data"
# replace the /Users/ppuente/github/detection_comparison-CRB/comparison-env to your own path and what you named the your environment (ie. comparison-env)

In [60]:
#packages
import geopandas as gpd
import rasterstats


huc4_path = '../data/shapefiles/crb_huc4.shp'

crb_huc4 = gpd.read_file(huc4_path)

#list of files 
file_list = ['../data/zenodo-data/landsat_mode_inundation.tif',
             '../data/zenodo-data/sentinel_extent_inundation.tif']

product_name = ['landsat', 'sentinel']

zonal_data = {
    'name': [],             #name of product
    'huc4': [],             #huc4 id
    'count_inundated': [],  #number of inundated pixels 
    'count_huc': []         #number of total pixels in the huc, all classifications 0 & 1

}

In [61]:
# perform the zonal statistics
counter = 0 
for file in file_list:
    count_water = rasterstats.zonal_stats(huc4_path, file,
                                          stats = ['count'],
                                          categorical = True, 
                                          geojson_out = True)
    
    print(f'zonal stats calculated for, {product_name[counter]}')

    #get the hucs in order 
    huc4_order = []
    for i in range(0, len(count_water)):
        huc4_id = count_water[i]['properties']['huc4']
        huc4_order.append(int(huc4_id))

    sorted_huc4_i = np.argsort(huc4_order)

    for i in sorted_huc4_i:
        name = product_name[counter]
        huc4_id = count_water[i]['properties']['huc4']
        count_inund = count_water[i]['properties'][1]
        count_huc = count_water[i]['properties']['count']

        #save to data 
        zonal_data['name'].append(name)
        zonal_data['huc4'].append(huc4_id)
        zonal_data['count_inundated'].append(count_inund)
        zonal_data['count_huc'].append(count_huc)

    print(f'completed process for {product_name[counter]}')
    counter += 1

/Users/ppuente/github/detection_comparison-CRB/comparison-env/lib/python3.11/site-packages/rasterstats/io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


zonal stats calculated for, landsat
completed process for landsat
zonal stats calculated for, sentinel
completed process for sentinel


In [63]:
import pandas as pd 

zonal_df = pd.DataFrame(zonal_data)

########
# Calculating Area 
#  (number of pixels) * (pixel area = 10x10 m^2) / 1e6 to convert to km 
########

#size of pixel 
pixel_area = 10**2 #(meters^2) from literature
#convert to kilometers
km_scale = 1e6

#add calculations of area 
zonal_df['area_inundated'] = (zonal_df.count_inundated * pixel_area) / km_scale

In [64]:
zonal_df

,name,huc4,count_inundated,count_huc,area_inundated
0,landsat,1401,2575603,429629292,257.5603
1,landsat,1402,1459827,340589925,145.9827
2,landsat,1403,933596,352843392,93.3596
3,landsat,1404,7230331,969547109,723.0331
4,landsat,1405,1336071,593939798,133.6071
5,landsat,1406,4156213,637829206,415.6213
6,landsat,1407,6353877,565115928,635.3877
7,landsat,1408,2830520,1009309172,283.0520
8,landsat,1501,6664098,1222043022,666.4098
9,landsat,1502,1203118,1053204694,120.3118


In [65]:
zonal_df.to_csv('../data/huc4_zonal_stats.csv', index=False)

### Counting the number of pixels for the inundation of the 3 cases of overlap between Sentinel and Landsat

- both inundated: Landsat and Sentinel pixel overlap are both inundated 
- landsat only: Landsat pixel is iundated and Sentinel pixel is not
- sentinel only: Sentinel pixel is iundated and Landsat pixel is not

In [2]:
import rasterio 
import numpy as np

huc4_raster_path = '../data/zenodo-data/huc4_raster.tif'

tif_files = ['../data/zenodo-data/both_inundated.tif',
            '../data/zenodo-data/landsat_only.tif',
            '../data/zenodo-data/sentinel_only.tif']

tif_names = [tif_path.split('/')[-1].replace('.tif', '') for tif_path in tif_files]  # Remove .tif extension for column names

with rasterio.open(huc4_raster_path) as src:
    huc_raster = src.read(1)

In [3]:
#function that counts pixels without loading the full raster 
def count_pixels(src, huc_mask, valid_values):
    # Get only the bounds of the HUC region
    rows, cols = np.where(huc_mask)
    if len(rows) == 0:  # No pixels found
        return 0

    # Define the bounding box of the valid pixels
    row_min, row_max = rows.min(), rows.max()
    col_min, col_max = cols.min(), cols.max()
    window = rasterio.windows.Window(col_min, row_min, col_max - col_min + 1, row_max - row_min + 1)

    # Read only this window from the raster
    raster_chunk = src.read(1, window=window)

    # Mask the HUC area inside the window
    local_huc_mask = huc_mask[row_min:row_max + 1, col_min:col_max + 1]
    valid_pixels = np.isin(raster_chunk[local_huc_mask], valid_values)

    return np.sum(valid_pixels)  # Count matching pixels

In [4]:
#reverse the mapping we did in the pre-processing of hucs due to memory 
unique_values = [1,2,3,4,5,6,7,8,11,12,13,14,15,16,17]
huc_labels = [1401, 1402, 1403, 1404, 1405, 1406, 1407, 1408, 1501, 1502, 1503, 1504, 1505, 1506, 1507]
#collect the results
save_results = {huc_label: {'huc4': huc_label} for huc_label in huc_labels}


In [5]:
#instead lets call the raster once and count the pixels 
for tif_path, tif_name in zip(tif_files, tif_names): 
    print(f'Processing for {tif_name}...')

    #read the full raster only once 
    with rasterio.open(tif_path) as src:
        #raster_data = src.read(1) 
    
        for huc_id, huc_label in zip(unique_values, huc_labels): 
            huc_mask = (huc_raster == huc_id) # Create mask once per HUC

            count = count_pixels(src, huc_mask, [1])
            
            save_results[huc_label][tif_name] = count

            print(f'Calculated for huc {huc_label}')

Processing for both_inundated...
Calculated for huc 1401
Calculated for huc 1402
Calculated for huc 1403
Calculated for huc 1404
Calculated for huc 1405
Calculated for huc 1406
Calculated for huc 1407
Calculated for huc 1408
Calculated for huc 1501
Calculated for huc 1502
Calculated for huc 1503
Calculated for huc 1504
Calculated for huc 1505
Calculated for huc 1506
Calculated for huc 1507
Processing for landsat_only...
Calculated for huc 1401
Calculated for huc 1402
Calculated for huc 1403
Calculated for huc 1404
Calculated for huc 1405
Calculated for huc 1406
Calculated for huc 1407
Calculated for huc 1408
Calculated for huc 1501
Calculated for huc 1502
Calculated for huc 1503
Calculated for huc 1504
Calculated for huc 1505
Calculated for huc 1506
Calculated for huc 1507
Processing for sentinel_only...
Calculated for huc 1401
Calculated for huc 1402
Calculated for huc 1403
Calculated for huc 1404
Calculated for huc 1405
Calculated for huc 1406
Calculated for huc 1407
Calculated for h

Make a note here that it crashed, load the data of the output that its supposed to give, turn the output of the past file into what we want here (i.e. only keep the columns needed). 

fix this later

In [31]:
import pandas as pd
# Convert dictionary to DataFrame
df_results = pd.DataFrame.from_dict(save_results, orient='index')

# Save to CSV
#df_results.to_csv("../data/huc4_agreement_pixel_counts.csv", index=False)

print("Processing complete. Results saved to huc_pixel_counts.csv")

Processing complete. Results saved to huc_pixel_counts.csv


In [32]:
df_results

,huc4,both_inundated,landsat_only,sentinel_only
1401,1401,0,121025,1804213
1402,1402,0,100566,1145089
1403,1403,0,67146,2186198
1404,1404,0,520796,3106628
1405,1405,0,112332,1721077
1406,1406,0,215345,3388844
1407,1407,0,299828,2466027
1408,1408,0,244682,2436213
1501,1501,0,220072,12506278
1502,1502,0,459777,1043453


In [33]:
df_results = df_results.drop('both_inundated', axis = 1)

In [34]:
df_results

,huc4,landsat_only,sentinel_only
1401,1401,121025,1804213
1402,1402,100566,1145089
1403,1403,67146,2186198
1404,1404,520796,3106628
1405,1405,112332,1721077
1406,1406,215345,3388844
1407,1407,299828,2466027
1408,1408,244682,2436213
1501,1501,220072,12506278
1502,1502,459777,1043453


In [37]:
#both_inundated did not count, alternatively adding column from previous run 

alt_data = pd.read_csv('/Users/ppuente/github/product_comparison_SW/data/huc4_agreement_pixel_counts.csv')

alt_data
#both_inundated = alt_data['mode_inundated_agreement']

#both_inundated

#add column to file 
#df_results['both_inundate'] = alt_data['mode_inundated_agreement']

,huc4,extent_inundated_agreement,extent_dry_agreement,extent_dry_landsat_inundated_sentinel,extent_inundated_landsat_dry_sentinel,mode_inundated_agreement,mode_dry_agreement,mode_dry_landsat_inundated_sentinel,mode_inundated_landsat_dry_sentinel
0,1401,2600173,424919552,1658658,444619,2454618,425243146,1804213,121025
1,1402,1420974,337922341,1083339,158753,1359224,337980528,1145089,100566
2,1403,954385,349637543,2098263,151657,866450,349722054,2186198,67146
3,1404,6949765,958136271,2866401,1573525,6709538,959189001,3106628,520795
4,1405,1334094,590760527,1610735,229571,1223752,590877766,1721077,112332
5,1406,4135761,629795902,3193920,690671,3940837,630271228,3388844,215345
6,1407,6238220,555563985,2281806,987265,6053999,556251431,2466027,299819
7,1408,2766764,1003481708,2255361,744659,2585912,1003981687,2436213,244680
8,1501,6930515,1201613592,12019778,1221324,6444015,1202614844,12506278,220072
9,1502,953987,1049813389,832839,1526478,743373,1050880090,1043453,459777


In [35]:
# Concatenate df1['B'] as a new column to df_new
df_new = pd.concat([df_results, alt_data['mode_inundated_agreement'].rename('both_inundated')], axis=1)


In [36]:
df_new

,huc4,landsat_only,sentinel_only,both_inundated
1401,1401.0,121025.0,1804213.0,NaN
1402,1402.0,100566.0,1145089.0,NaN
1403,1403.0,67146.0,2186198.0,NaN
1404,1404.0,520796.0,3106628.0,NaN
1405,1405.0,112332.0,1721077.0,NaN
1406,1406.0,215345.0,3388844.0,NaN
1407,1407.0,299828.0,2466027.0,NaN
1408,1408.0,244682.0,2436213.0,NaN
1501,1501.0,220072.0,12506278.0,NaN
1502,1502.0,459777.0,1043453.0,NaN
